# Fast No RL 02 - Optional Supervised Learning

Optional: train the existing policy/value architecture against labelled positions, without self-play. This is supervised deep learning, not a requirement for the classical baseline. Set variant to `supervised` before running. An A100/CUDA runtime accelerates this optional notebook only; inference remains CPU.

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project Setup

Uses `configs/fast_no_rl.yaml`. No league, self-play, PPO, or DQN is run. Colab's installed PyTorch is preserved.

In [ ]:
from pathlib import Path
import sys
import subprocess
from IPython.display import display
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

def show_figure(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl/non_rl.py").is_file():
    raise FileNotFoundError(f"Place the updated project contents directly in {PROJECT_ROOT}")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
                       str(PROJECT_ROOT / "requirements_colab.txt")])
sys.path.insert(0, str(PROJECT_ROOT))
from chess_rl.non_rl import load_no_rl_config
from chess_rl.reproducibility import read_json, sha256
cfg = load_no_rl_config(PROJECT_ROOT, "fast_no_rl.yaml")
print("Run:", cfg["run_id"], "| Variant:", cfg["non_rl"]["variant"])
print("Root:", PROJECT_ROOT)


## Training Choice

Every expensive cell is disabled for the default classical variant. The default supervised model has six 128-channel residual blocks, a 4,672-action policy head, and one value output. Defaults: up to 20 epochs, AdamW, learning rate 0.0003, cosine scheduling, and validation early stopping.

In [ ]:
TRAIN_SUPERVISED = cfg["non_rl"]["variant"] == "supervised"
print("Supervised training enabled:", TRAIN_SUPERVISED)
if TRAIN_SUPERVISED:
    from chess_rl.reproducibility import seed_all, resolve_device
    seed_all(cfg["seed"], cfg["deterministic"])
    print("Training device:", resolve_device(cfg["device"]))
    print("Architecture:", cfg["model"])
else:
    print("Skip this notebook. Open no-RL notebook 03.")

## Offline Teacher

Stockfish is used only to label offline data. It is not shipped in the candidate. The configured classical teacher is also supported. This setup runs only for the supervised variant.

In [ ]:
if TRAIN_SUPERVISED:
    teacher_path = Path(cfg["dataset"]["engine_path"])
    if cfg["dataset"]["teacher"] == "stockfish" and not teacher_path.is_file():
        subprocess.check_call(["apt-get", "update", "-qq"])
        subprocess.check_call(["apt-get", "install", "-y", "-qq", "stockfish"])
    if cfg["dataset"]["teacher"] == "stockfish":
        if not teacher_path.is_file():
            raise FileNotFoundError(f"Configure dataset.engine_path: {teacher_path}")
        print("Teacher hash:", sha256(teacher_path))

## Build or Resume the Dataset

Provide PGNs or JSONL through a `dataset` override in `configs/fast_no_rl.yaml`. Empty broad sources now stop with a clear error instead of generating synthetic training positions. This is not a curated strategy curriculum. Source-game splits and held-out exclusion use the shared dataset implementation.

In [ ]:
if TRAIN_SUPERVISED:
    from chess_rl.dataset import prepare_dataset, prepare_openings
    prepare_openings(PROJECT_ROOT, seed=cfg["seed"])
    dataset_manifest = prepare_dataset(PROJECT_ROOT, cfg)
    print("Actual split counts:", dataset_manifest["counts"])
    print("Target reached:", dataset_manifest["target_reached"])

## Train and Save

Policy targets are legal teacher moves; value targets are teacher evaluations or completed-game results, with their sources distinguished. Training uses legal-only cross-entropy plus value Huber loss. Latest/best checkpoints, optimizer state, RNG, config, and CSV logs are saved. Interrupted epochs resume from the last completed epoch.

In [ ]:
if TRAIN_SUPERVISED:
    from chess_rl.training import fit_supervised
    checkpoint = fit_supervised(PROJECT_ROOT, cfg, dataset_manifest)
    print("Best supervised checkpoint:", checkpoint)
    print("SHA256:", sha256(checkpoint))

## Inspect Learning Curves

These plots use actual completed-epoch logs only. Lower label loss does not establish stronger play. Move on to no-RL notebook 03 to measure games; no self-play stage follows this notebook.

In [ ]:
if TRAIN_SUPERVISED:
    from chess_rl.plots import plot_supervised
    show_figure(plot_supervised(PROJECT_ROOT, cfg["run_id"]))